In [0]:
%sql
select * from read_files('/Volumes/idp/default/final_project')

In [0]:
%sql
create or replace table parsed_data as
select path,
ai_parse_document(content) as parsed_content
from read_files('/Volumes/idp/default/final_project')

In [0]:
%sql
select * from parsed_data

In [0]:
%sql
create or replace table pretty_data as
SELECT path,
  concat_ws('\n',
    transform(try_cast(parsed_content:document:elements AS ARRAY<VARIANT>), e -> coalesce(try_cast(e:content AS STRING), ''))
  ) as doc_text
FROM parsed_data

In [0]:
%sql
select * from pretty_data

In [0]:
%sql
create or replace table classified_data as
SELECT *,
  ai_classify(doc_text, ARRAY('Invoice','Purchase Order','Receipt','Other')) as doc_classification
FROM pretty_data

In [0]:
%sql
select * from classified_data
where doc_classification= 'Invoice'

In [0]:
%sql
CREATE or REPLACE table invoice_data as
SELECT *,
  ai_extract(doc_text,
    ARRAY('Vendor_Name',
          'Invoice_Number',
          'Invoice_Date',
          'Due_Date',
          'Payment_Method',
          'Total')) as extracted
FROM classified_data
WHERE doc_classification = 'Invoice'

In [0]:
%sql
select * from invoice_data

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS idp.finance

In [0]:
%sql
 CREATE OR REPLACE TABLE idp.finance.invoices AS
 SELECT path,
  extracted.Vendor_Name AS Vendor,
  extracted.Invoice_Number AS Invoice_Number,
  extracted.Invoice_Date AS Invoice_Date,
  extracted.Due_Date AS Due_Date,
  extracted.Payment_Method AS Payment_Method,
  extracted.Total AS Total
FROM invoice_data;


In [0]:
%sql
select * from idp.finance.invoices

In [0]:
%sql
CREATE OR REPLACE TABLE purchase_order_data AS
SELECT *,
ai_extract(doc_text,
    ARRAY('Merchant_Name',
          'PO_Number',
          'Purchase_Order_Date',
          'Total')) as extracted
FROM classified_data
WHERE doc_classification = 'Purchase Order'

In [0]:
%sql
select * from purchase_order_data

In [0]:
%sql
CREATE OR REPLACE TABLE idp.finance.purchase_order AS
SELECT path,
  extracted.Merchant_Name AS Merchant,
  extracted.PO_Number AS PO_Number,
  extracted.Purchase_Order_Date AS Purchase_Order_Date,
  extracted.Total AS Total
FROM purchase_order_data

In [0]:
%sql
CREATE OR REPLACE TABLE Receipt AS
SELECT *,
  ai_extract(doc_text,
    ARRAY('Merchant_Name',
          'Receipt_Number',
          'Transaction_Date',
          'Total')) as extracted
FROM classified_data
WHERE doc_classification = 'Receipt'

In [0]:
%sql
CREATE OR REPLACE TABLE idp.finance.receipt AS
SELECT path,
  extracted.Merchant_Name AS Merchant,
  extracted.Receipt_Number AS Receipt_Number,
  extracted.Transaction_Date AS Transaction_Date,
  extracted.Total AS Total
FROM receipt